In [127]:
import pandas as pd

In [128]:
df = pd.read_csv('../data/reference.csv')

### NULLS

In [129]:
## percentage of null values in each column
percent_nulos = (df.isnull().mean() * 100).round(2)
print(percent_nulos)

SK_ID_CURR                     0.00
TARGET                         0.00
NAME_CONTRACT_TYPE             0.00
CODE_GENDER                    0.00
FLAG_OWN_CAR                   0.00
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     13.51
AMT_REQ_CREDIT_BUREAU_WEEK    13.51
AMT_REQ_CREDIT_BUREAU_MON     13.51
AMT_REQ_CREDIT_BUREAU_QRT     13.51
AMT_REQ_CREDIT_BUREAU_YEAR    13.51
Length: 122, dtype: float64


In [130]:
## percentage of null values in each column
percent_nulos.sort_values(ascending=False).head(20)


COMMONAREA_AVG              69.84
COMMONAREA_MODE             69.84
COMMONAREA_MEDI             69.84
NONLIVINGAPARTMENTS_MEDI    69.40
NONLIVINGAPARTMENTS_MODE    69.40
NONLIVINGAPARTMENTS_AVG     69.40
FONDKAPREMONT_MODE          68.38
LIVINGAPARTMENTS_AVG        68.34
LIVINGAPARTMENTS_MEDI       68.34
LIVINGAPARTMENTS_MODE       68.34
FLOORSMIN_MODE              67.85
FLOORSMIN_AVG               67.85
FLOORSMIN_MEDI              67.85
YEARS_BUILD_AVG             66.48
YEARS_BUILD_MODE            66.48
YEARS_BUILD_MEDI            66.48
OWN_CAR_AGE                 66.02
LANDAREA_MEDI               59.34
LANDAREA_AVG                59.34
LANDAREA_MODE               59.34
dtype: float64

In [131]:
## quantity of columns with null values
null = df.isnull().sum() != 0
null_columns = df.columns[null]
qtt_null = len(null_columns)
qtt_null

67

In [132]:
correlacao_target = (
    df.corr(numeric_only=True)['TARGET']
    .drop('TARGET')
    .sort_values(ascending=False)
)

display(correlacao_target)

DAYS_BIRTH                     0.079467
REGION_RATING_CLIENT_W_CITY    0.061283
REGION_RATING_CLIENT           0.058946
DAYS_LAST_PHONE_CHANGE         0.053795
REG_CITY_NOT_WORK_CITY         0.051666
                                 ...   
FLOORSMAX_AVG                 -0.043707
DAYS_EMPLOYED                 -0.045355
EXT_SOURCE_1                  -0.155362
EXT_SOURCE_2                  -0.159593
EXT_SOURCE_3                  -0.178845
Name: TARGET, Length: 105, dtype: float64

### Corr

In [133]:
## correlation of columns with TARGET
correlacao_target.sort_values(ascending=False).head(20)

DAYS_BIRTH                     0.079467
REGION_RATING_CLIENT_W_CITY    0.061283
REGION_RATING_CLIENT           0.058946
DAYS_LAST_PHONE_CHANGE         0.053795
REG_CITY_NOT_WORK_CITY         0.051666
DAYS_ID_PUBLISH                0.049582
FLAG_EMP_PHONE                 0.046407
REG_CITY_NOT_LIVE_CITY         0.045654
FLAG_DOCUMENT_3                0.044299
DAYS_REGISTRATION              0.042127
OWN_CAR_AGE                    0.038289
LIVE_CITY_NOT_WORK_CITY        0.033339
DEF_60_CNT_SOCIAL_CIRCLE       0.032074
DEF_30_CNT_SOCIAL_CIRCLE       0.032059
FLAG_WORK_PHONE                0.028306
CNT_CHILDREN                   0.020347
AMT_REQ_CREDIT_BUREAU_YEAR     0.018984
OBS_30_CNT_SOCIAL_CIRCLE       0.010773
OBS_60_CNT_SOCIAL_CIRCLE       0.010617
CNT_FAM_MEMBERS                0.009804
Name: TARGET, dtype: float64

In [134]:
## correlation of columns with TARGET
df[['COMMONAREA_AVG', 'COMMONAREA_MODE',
    'COMMONAREA_MEDI', 'TARGET']].corr()['TARGET']

COMMONAREA_AVG    -0.015986
COMMONAREA_MODE   -0.013730
COMMONAREA_MEDI   -0.015961
TARGET             1.000000
Name: TARGET, dtype: float64

In [135]:
df.groupby('TARGET')['COMMONAREA_AVG'].describe()

,count,mean,std,min,25%,50%,75%,max
TARGET,,,,,,,,
0,69037.0,0.044943,0.076155,0.0,0.0079,0.0213,0.0520,1.0
1,5160.0,0.040184,0.069587,0.0,0.0073,0.0189,0.0469,1.0


In [136]:
df[['COMMONAREA_AVG', 'COMMONAREA_MODE', 'COMMONAREA_MEDI']].corr()

,COMMONAREA_AVG,COMMONAREA_MODE,COMMONAREA_MEDI
COMMONAREA_AVG,1.000000,0.975988,0.995660
COMMONAREA_MODE,0.975988,1.000000,0.978934
COMMONAREA_MEDI,0.995660,0.978934,1.000000


In [137]:
colunms = [
    'COMMONAREA_AVG',
    'COMMONAREA_MODE',
    'COMMONAREA_MEDI'
]

for colunm in colunms:
    print(f'\nAnálise: {colunm}')

    result = df.assign(
        MISSING=df[colunm].isnull().astype(int)
    ).groupby('MISSING')['TARGET'].agg(
        ['count', 'mean']
    )

    display(result)


Análise: COMMONAREA_AVG


,count,mean
MISSING,,
0,74197,0.069545
1,171811,0.085559



Análise: COMMONAREA_MODE


,count,mean
MISSING,,
0,74197,0.069545
1,171811,0.085559



Análise: COMMONAREA_MEDI


,count,mean
MISSING,,
0,74197,0.069545
1,171811,0.085559


In [138]:
##selecting columns that end with "_AVG", "_MODE", or "_MEDI"
housing_cols = df.columns[df.columns.str.endswith(("_AVG", "_MODE", "_MEDI"))]
print(len(housing_cols))

47


In [139]:
df["HOUSING_INFO_MISSING"] = df[housing_cols].isnull().all(axis=1).astype(int)

In [140]:
df.groupby("HOUSING_INFO_MISSING")["TARGET"].agg(["count", "mean"])

,count,mean
HOUSING_INFO_MISSING,,
0,129427,0.070109
1,116581,0.092519


### Decision: housing characteristics columns

**What:** dropped the 47 columns with suffixes `_AVG`, `_MODE` and `_MEDI`;
created the indicator `HOUSING_INFO_MISSING` (1 = all 47 columns are null).

**Why:**
- Redundancy: the three versions of each characteristic have correlation > 0.97.
- Weak signal from values: correlation with TARGET of about −0.016.
- High missingness: between 59% and 70% null values.
- Informative missingness: default rate of 9.25% (no information) vs. 7.01% (information available).

**Caveat:** the indicator shows association, not causation.

In [141]:
df = df.drop(columns=housing_cols)

In [142]:
df.columns

Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE',
       'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
       'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE',
       'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE',
       'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS',
       'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION',
       'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY',
       'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
       'OBS_3

### Car

In [143]:
## Cross-tabulation of car ownership and missing car age
car = pd.crosstab(
    df['FLAG_OWN_CAR'],
    df['OWN_CAR_AGE'].isnull()
)
print(car)

OWN_CAR_AGE   False   True 
FLAG_OWN_CAR               
N                 0  162413
Y             83590       5


In [144]:
## Evidence: count and default rate per group
df.groupby("FLAG_OWN_CAR")["TARGET"].agg(["count", "mean"])

,count,mean
FLAG_OWN_CAR,,
N,162413,0.084839
Y,83595,0.072744


In [145]:
# Group car age into fixed bins; clients without a car get their own category
car_age_group = pd.cut(
    df["OWN_CAR_AGE"],
    bins=[0, 5, 10, 15, 20, float("inf")],
    labels=["0-5", "5-10", "10-15", "15-20", "20+"],
    include_lowest=True,
)

df["CAR_AGE_GROUP"] = car_age_group.cat.add_categories("NO_CAR").fillna("NO_CAR")

In [146]:
## Evidence: count and default rate per group
df.groupby("CAR_AGE_GROUP", observed=True)["TARGET"].agg(["count", "mean"])

,count,mean
CAR_AGE_GROUP,,
0-5,23009,0.056978
5-10,23551,0.063819
10-15,16928,0.082349
15-20,9616,0.092554
20+,10486,0.093744
NO_CAR,162418,0.084837


In [147]:
## Evidence: count of car age values
df['OWN_CAR_AGE'].value_counts(dropna=False).sort_index()

OWN_CAR_AGE
0.0       1732
1.0       4164
2.0       4683
3.0       5109
4.0       4457
         ...  
63.0         2
64.0      1942
65.0       694
91.0         2
NaN     162418
Name: count, Length: 62, dtype: int64

### Decision: OWN_CAR_AGE → CAR_AGE_GROUP

**What:** grouped car age into fixed bins (`0-5`, `5-10`, `10-15`, `15-20`, `20+`)
and created the category `NO_CAR` for clients without a car.
`FLAG_OWN_CAR` is kept for the data contract but dropped from the model (redundant).

**Why:**
- Nulls are structural: all 162,413 clients with `FLAG_OWN_CAR = N` have a null car age.
- Car age separates risk more than car ownership: default rate rises from 5.7% (0-5 years)
  to 9.4% (20+ years), while the flag alone separates 7.3% (Y) vs. 8.5% (N).
- Fixed bins (not quantiles) keep drift visible in production.

**Data quality findings:**
- 5 records have `FLAG_OWN_CAR = Y` and a null car age → blocked by the contract.
- Suspicious peaks at 64 years (1,942 records) and 65 years (694), with only 2 records
  at 63 → alert rule for car age above 60.

**Caveat:** the peaks look like a default value, but their origin could not be confirmed.

### Age

In [148]:

df['DAYS_BIRTH'].describe()

count    246008.000000
mean     -16029.515882
std        4362.869037
min      -25229.000000
25%      -19673.000000
50%      -15744.000000
75%      -12401.000000
max       -7489.000000
Name: DAYS_BIRTH, dtype: float64

In [149]:
# Convert days before application to age in years (negative days → positive age)
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).round(1)
df['AGE_YEARS'].describe()


count    246008.000000
mean         43.886490
std          11.944872
min          20.500000
25%          34.000000
50%          43.100000
75%          53.900000
max          69.100000
Name: AGE_YEARS, dtype: float64

### Decision: DAYS_BIRTH → AGE_YEARS

**What:** created `AGE_YEARS = -DAYS_BIRTH / 365.25`. The raw `DAYS_BIRTH` is kept
unchanged, because the data contract validates it.

**Why:**
- `DAYS_BIRTH` counts days before the application date, so all values are negative.
- Age range in the reference: 20.5 to 69.1 years (median 43.1).
- Contract rule: age between 18 and 115. No reference record violates it, so the rule
  is proven with a deliberately invalid batch.

**Caveat:** age is kept as a model feature because it relates to credit risk,
but it is included in the bias audit (decisions are compared across age groups).

### Days Employed

In [150]:
df['AMT_INCOME_TOTAL'].describe()

count    2.460080e+05
mean     1.688532e+05
std      2.589606e+05
min      2.565000e+04
25%      1.125000e+05
50%      1.476000e+05
75%      2.025000e+05
max      1.170000e+08
Name: AMT_INCOME_TOTAL, dtype: float64

In [151]:
df['DAYS_EMPLOYED'].describe()

count    246008.000000
mean      63581.163637
std      141081.696923
min      -17912.000000
25%       -2762.000000
50%       -1215.000000
75%        -291.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

In [152]:
# Convert 365243 days to years
365243 / 365.25

999.98083504449

In [153]:
## Count of clients with 365243 days employed
(df["DAYS_EMPLOYED"] == 365243).sum()

np.int64(44143)

In [154]:
## Count of clients with 365243 days employed by income type
df.loc[df["DAYS_EMPLOYED"] == 365243, "NAME_INCOME_TYPE"].value_counts()

NAME_INCOME_TYPE
Pensioner     44126
Unemployed       17
Name: count, dtype: int64

In [155]:
## Convert days employed to years employed, treating 365243 as missing
years_employed = (-df["DAYS_EMPLOYED"] / 365.25).where(df["DAYS_EMPLOYED"] != 365243)

In [156]:
## Group years employed into fixed bins; clients with 365243 days employed get their own category
employment_group = pd.cut(
    years_employed,
    bins=[0, 1, 3, 5, 10, float("inf")],
    labels=["0-1", "1-3", "3-5", "5-10", "10+"],
    include_lowest=True,
)

In [157]:
## Evidence: count and default rate per group
df["EMPLOYMENT_GROUP"] = (
    employment_group.cat.add_categories("NOT_EMPLOYED").fillna("NOT_EMPLOYED")
)

In [158]:

df.groupby("EMPLOYMENT_GROUP", observed=True)["TARGET"].agg(["count", "mean"])

,count,mean
EMPLOYMENT_GROUP,,
0-1,22393,0.109945
1-3,49260,0.110617
3-5,37558,0.096225
5-10,51877,0.074330
10+,40777,0.051720
NOT_EMPLOYED,44143,0.053689


### Decision: DAYS_EMPLOYED → EMPLOYMENT_GROUP

**What:** converted DAYS_EMPLOYED (days since the start of the current job,
relative to the application date) into years and grouped it into fixed bins:
`0-1`, `1-3`, `3-5`, `5-10`, `10+`. The value 365243 (~1,000 years) was mapped
to its own category, `NOT_EMPLOYED`.

**Why:**
- 365243 is a sentinel code, not a real duration: 44,143 records (17.9%) share
  exactly this value, all of them Pensioner (44,126) or Unemployed (17).
- Default rate falls as tenure increases: 11.0% (0-1 years) → 5.2% (10+ years).
- NOT_EMPLOYED has a low default rate (5.4%). Filling the code with 0 years would
  merge it with the highest-risk group (0-1 years, 11.0%) and distort the signal.

**Caveat:** the meaning of 365243 was inferred from the data; the dataset does
not officially document it.

### Income

In [159]:
cols = [
    "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "OCCUPATION_TYPE",
    "EMPLOYMENT_GROUP", "AMT_CREDIT", "AMT_ANNUITY", "TARGET",
]
df.loc[df["AMT_INCOME_TOTAL"] == df["AMT_INCOME_TOTAL"].max(), cols]

,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,EMPLOYMENT_GROUP,AMT_CREDIT,AMT_ANNUITY,TARGET
86284,117000000.0,Working,Laborers,1-3,562491.0,26194.5,1


In [160]:
df["AMT_INCOME_TOTAL"].min()

np.float64(25650.0)

### Decision: AMT_INCOME_TOTAL

**What:** no transformation. Contract rules: income must be non-null and greater
than zero (block); unusually high income and annuity above income (alert).

**Why:**
- No nulls in the reference; minimum 25,650; median 147,600.
- Maximum of 117,000,000 (~800x the median). Context suggests a typing error:
  laborer, 1-3 years employed, annuity of 26,194.5 (0.02% of the income), defaulted.
- The value is implausible but not impossible, so it is flagged, not blocked.

**Caveat:** the dataset does not document the currency or whether income is
monthly or annual; values are compared only relative to each other.

### Decision: CODE_GENDER

**What:** `CODE_GENDER` is excluded from the model features. It is kept in the
dataset, separately from the features, only for the bias audit.

**Why:**
- Non-discrimination principle (LGPD, art. 6, IX): a client's sex should not
  influence the credit decision.
- Removing the column from the model does not remove the bias. Other features
  may act as proxies for sex (e.g., family status, occupation, income type).
- Keeping the column outside the model makes it possible to measure whether the
  model treats men and women differently (approval and error rates by group).

**Caveat:** excluding the attribute ("fairness through unawareness") does not
guarantee a fair model. Fairness must be measured in the bias audit, and access
to this column is restricted to that purpose.